# 第21章：完成端到端业务任务

本篇依据[本章课件](https://github.com/xbsheng/atguigu-note/blob/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E8%AF%BE%E4%BB%B6/%E5%AE%9E%E6%88%98%E7%AF%87/07-%E5%AE%8C%E6%88%90%E7%AB%AF%E5%88%B0%E7%AB%AF%E4%B8%9A%E5%8A%A1%E4%BB%BB%E5%8A%A1.md)整理，对应[视频 P32](https://www.bilibili.com/video/BV14rxM6qEnL?p=32)。资料快照：`bbe79814defe`（2026-10-29）。

以下命令、提示词和示例为课程阅读内容；样本数字是预期核对值，未在本次笔记整理中运行课程服务、安装技能或执行审批。

[完整配套工程](https://github.com/xbsheng/atguigu-note/tree/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E4%BB%A3%E7%A0%81/ecommerce-agent)；[业务口径](https://github.com/xbsheng/atguigu-note/blob/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E8%AF%BE%E4%BB%B6/%E5%AE%9E%E6%88%98%E7%AF%87/%E8%B5%84%E6%96%99/%E4%B8%9A%E5%8A%A1%E9%9C%80%E6%B1%82%E4%B8%8E%E6%95%B0%E6%8D%AE%E5%8F%A3%E5%BE%84.md)；[验收场景](https://github.com/xbsheng/atguigu-note/blob/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E8%AF%BE%E4%BB%B6/%E5%AE%9E%E6%88%98%E7%AF%87/%E8%B5%84%E6%96%99/%E9%AA%8C%E6%94%B6%E5%9C%BA%E6%99%AF%E4%B8%8E%E9%A2%84%E6%9C%9F%E7%BB%93%E6%9E%9C.md)。

## 21.1 串起一次完整业务任务

已有系统 → MCP六工具 → 业务Skill → LangGraph暂停/恢复 → 工单与报告。开始前API和MCP已运行，Codex能发现技能和工具。固定店铺A、付款日2026-10-10、基准2026-10-11T12:00:00+08:00。

## 21.2 Agent与人工审批分开

Agent凭据用于查询推进，人工审批凭据只给独立审批入口，不加入聊天、MCP、Codex环境和成果。教程同机.env仅演示角色边界，企业须在服务端隔离真实凭据。

流程为：用户要求 → Agent启动 → 服务端返回报告并暂停 → 人工独立决定 → Agent查询决定并恢复 → 返回真实工单结果。审批与执行是两个步骤。

## 21.3 发起工作流

在Codex对话明确使用ecommerce-order-investigation，给固定时间、严格24小时条件、库存/履约证据和建单前等待审批，request_id用 `demo-a-20261010-001`。要求返回workflow_id、状态和完整report。

同一请求网络重试保留原request_id，条件变更另开新编号。workflow_id必须取服务端返回，不能根据订单号猜测。

### 一轮请求要保留哪些输入

业务请求应明确店铺、付款日、带时区基准、严格24小时条件和是否需要处理工单，并给本次request_id。Agent根据返回展示真实workflow_id与report，不用订单号猜流程号。

如果本轮只要报告，应走只读查询；这里明确要启动处理流程，因此返回待审批状态就是预期中间成果，不是任务失败。后续还需真实人工决定和恢复，不能在一条对话中把未来步骤写成已完成。

## 21.4 暂停前核对

A1001/2/3分别等28/27/26小时；证据依次是当前缺货、warehouse_hold、履约证据不足。A1004仅18小时，A1005/6已发货/取消，不计。

再查询工作流应为awaiting_approval，tickets为空。待处理建议没有变成数据库工单；调查可能原因不写成已完成修复。

## 21.5 人工决定与服务端记录

审批人阅读原流程清单，在工程目录独立终端选一个分支，替换实际id：

```powershell
uv run --frozen python -m app.approve 实际工作流编号 --decision approve
```

或不同意时：

```powershell
uv run --frozen python -m app.approve 实际工作流编号 --decision reject
```

隐藏输入审批令牌，核对服务端报告，再输入工作流编号确认；不要把令牌直接拼到命令行。此步只保存decision，尚未执行批准后的建单节点。

## 21.6 恢复后核对真实outcome

回Codex先查status确认决定，再用原workflow_id恢复，最后重新查询。初始数据不变且首次处理时，completed对应TK-A1001、TK-A1002、TK-A1003。

created表示本次新建，existing表示已有，skipped_changed表示审批期间订单已变化，不计成功建单。拒绝恢复为rejected，无异常为no_action。同流程重复恢复不增加工单，其他流程处理同订单也复用。

工单进入队列不代表库存补齐、订单发货或问题解决，报告保留业务真实状态。

### 将最后结果核对到服务端

审批后status先确认decision，resume执行，再status查看最终状态。初始三单首次处理才会出现三项created；若已有工单则existing，若一单已变状态则skipped_changed。不要用固定TK清单代替实际返回。

同流程重复恢复应不增加工单，另一流程同订单也复用。拒绝分支和无异常分支分别是rejected/no_action，各有不同业务含义，不能统一写“处理成功”。

## 21.7 保存可交接成果

从最终真实查询保存 `outputs/订单异常报告.md` 与 `订单异常结果.json`，有同名则用真实工作流id另命名。报告包含时间、范围、事实、推测、建议、状态和workflow_id；JSON保留服务返回。两份编号、数量、工单一一对应，状态变更后重新查询，不凭旧聊天推测。

### Markdown与JSON怎样相互核对

Markdown帮助接班人读范围、证据、建议与当前处理状态；JSON保留可供程序重用的原始结构。保存后逐一核对workflow_id、订单集合、outcome、工单id和决定，缺字段标明没有取得。

另存名称带真实流程id，可区分不同时间、不同审批的成果；接口状态变化之后重新查询生成新版本，而不是修改旧报告数字来假装反映最新状态。办公文件仍从同一份成果生成。

## 21.8 交给WorkBuddy制作办公文件

只传上述两份业务成果，由WorkBuddy生成可编辑Word报告和Excel异常明细。Word含范围、汇总、逐单证据与状态；Excel每单一行包含事实、推测、建议和工单id。未知保留待确认，表头、日期、列宽和换行只改变表达，不能改变JSON中的事实和审批状态。

## 21.9 变化输入与验收范围

提前as_of并用新request_id，应按时间重新筛选，不固定三单；B店查询仍不可访问；API失败说明查询失败，不能说零异常。完整交付串联业务请求、真实workflow_id、证据、人工决定和处理结果。

配套验收还包括：24小时边界、分页参数、空结果、无/错身份、格式校验、启动幂等及参数冲突、未审批恢复、Agent自批拒绝、批准/拒绝、重复恢复、已有工单、重启恢复、状态变化跳过、决定不可反写、网络失败和超100单拒绝。变化数据只在独立测试样本构造。

课件验证入口在工程目录：

```powershell
uv run --frozen pytest -q
uv run --frozen python -m app.smoke_mcp
```

API已启动初始化且当前终端已设置Agent环境时：

```powershell
uv run --frozen python -m app.smoke_mcp --with-api
```

命令成功不代表所有数据库和模型路线全覆盖；记录实际环境、输入、预期、原始响应/轨迹、判定及未执行项。这里列出教程验证方法，没有把这些课程系统测试标成已运行。

### 不同验证入口各自证明什么

离线测试覆盖字段、查询或状态机约束，smoke_mcp检查协议与工具返回；with-api还要求实际API与教学库已准备；真实模型请求则核对是否选对技能、读完分页、按证据解释并在审批点停下。

所以pytest通过不自动代表OpenAI和DeepSeek两条路线都通过，SQLite通过不自动代表MySQL部署通过。验收记录应写环境、样本、日期和请求id、预期、真实轨迹/响应及未运行项，让结果可复查。

变化输入验证复用：提前检查时刻应减少符合订单，用新request_id重新调查；恰好24小时不纳入；API不可达是失败；越权店铺仍不可见。固定三单只属于这组基准样本。